# Live spatial analysis: human lymph node
### WCI 2026 - Single cell/spatial transcriptomics course
### CSBL - PhD student Guilherme de-Mira

> ⚠️ **Instructor note — each student opens their own fresh Colab session from the QR code, so "run it ahead of time" only applies to your own demo copy, not theirs.** Build the timing into the class instead:
>
> 1. **You**: run this entire notebook once, start to finish, before class — so you have a fully executed reference on your own screen no matter what happens live.
> 2. **When you reveal the QR code to the room**: have everyone scan it, then immediately have them run only the install cell and the data-download cell (~1–1.5 min combined) — while that runs on ~30 laptops in parallel, keep talking (Section 1's dataset background is exactly the right length of content to cover here).
> 3. Section 12 (pathway enrichment) calls the public Enrichr web service over the internet for every student individually — normally instant, but one more thing that depends on the venue's wifi handling everyone at once.
> 4. Keep the **BACKUP_EXECUTED** copy of this notebook open on your own laptop as a fallback you can screen-share if the venue wifi struggles under everyone downloading at once.

In [ ]:
!pip install -q scanpy squidpy igraph gseapy

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
from matplotlib.collections import PolyCollection
from matplotlib.lines import Line2D

sc.settings.verbosity = 0
%matplotlib inline

## 1. The data: a real Xenium 5K human lymph node

This is a public dataset from 10x Genomics: a **reactive human lymph node**, profiled with the **Xenium Human 5K panel** (4,624 genes measured in situ, at single-cell resolution). It also carries each cell's real segmented outline, not just its center point — we'll use that below to see the cells as they actually look in the tissue.

In [ ]:
import os

DATA_URL = "https://github.com/guilhermemira266/xenium-lymph-node-demo/releases/download/v1/xenium_lymph_node_5k_crop.h5ad"
fname = "xenium_lymph_node_5k_crop.h5ad"

if not os.path.exists(fname):
    !wget -q "{DATA_URL}" -O "{fname}"

adata = sc.read_h5ad(fname)
adata

## 2. Quality control

In [ ]:
# Here, we filter out cells with less than 10 transcript reads, and genes
# detected in fewer than 3 cells.
sc.pp.filter_cells(adata, min_counts=10)
sc.pp.filter_genes(adata, min_cells=3)

# Before we perform normalization and log-transformation, we save the
# raw counts in a separate layer.
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

adata

## 3. Cluster cells by expression alone

> ⏱️ **Heads up, this cell takes ~30–40 seconds the first time you run it in a fresh session** — that's expected, not a crash. It isn't about the size of the data (8,677 cells is small); it's a one-time "warm-up" cost from a library compiling code the first time it's called in this session. Let it run. While it does, here's what's actually happening: PCA reduces the data to its main axes of variation, then we connect each cell to its nearest neighbors in that space, then Leiden groups cells that are densely connected — all without knowing anything about what each cell actually is.

In [ ]:
sc.pp.pca(adata, n_comps=30, random_state=0)
sc.pp.neighbors(adata, random_state=0)
sc.tl.leiden(adata, resolution=1.0, flavor="igraph", n_iterations=2, random_state=0)

adata.obs["leiden"].value_counts()

## Real cell shapes, not just dots

Every cell in this dataset also carries its actual segmented outline (a polygon), not just an x,y center point. Let's build those shapes once, and reuse them for every spatial plot below — a much more faithful picture of the tissue than a scatter of circles.

In [ ]:
boundaries = adata.uns["cell_boundaries"]
cell_polygons = {
    cid: g[["vertex_x", "vertex_y"]].values
    for cid, g in boundaries.groupby("cell_id", sort=False)
}
print(f"{len(cell_polygons):,} real cell shapes loaded")

BG_COLOR = "#0b0d12"
FILTERED_COLOR = "#20242c"  # cells dropped by QC, or without a label yet


def plot_by_boundary(color_col, palette, title, figsize=(7, 7)):
    labels = adata.obs[color_col]
    colors = [
        palette.get(labels[cid], FILTERED_COLOR) if cid in labels.index else FILTERED_COLOR
        for cid in cell_polygons
    ]
    fig, ax = plt.subplots(figsize=figsize, facecolor=BG_COLOR)
    ax.add_collection(PolyCollection(
        list(cell_polygons.values()), facecolors=colors,
        edgecolors="black", linewidths=0.15, alpha=0.85,
    ))
    ax.set_facecolor(BG_COLOR)
    ax.set_xlim(boundaries["vertex_x"].min(), boundaries["vertex_x"].max())
    ax.set_ylim(boundaries["vertex_y"].max(), boundaries["vertex_y"].min())
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title, color="white")

    handles = [
        Line2D([0], [0], marker="s", linestyle="none", markersize=10,
               markerfacecolor=c, markeredgecolor="none", label=k)
        for k, c in palette.items()
    ]
    handles.append(Line2D([0], [0], marker="s", linestyle="none", markersize=10,
                           markerfacecolor=FILTERED_COLOR, markeredgecolor="none", label="filtered / unlabeled"))
    ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.0, 1.0),
              frameon=False, labelcolor="white", fontsize=9)
    return fig, ax

## 4. Where do these clusters sit in the tissue?

Before we even know what these clusters *are*, let's just look at where they are.

In [ ]:
leiden_cats = adata.obs["leiden"].cat.categories
leiden_palette = {cat: plt.cm.tab10(i % 10) for i, cat in enumerate(leiden_cats)}

plot_by_boundary("leiden", leiden_palette, "leiden clusters — real cell shapes")

**Ask the audience**: *"You don't know what any of these numbered clusters means yet — but does it look random, or organized?"* Two clean, separate territories should already be visible. Let's find out what they are.

## 5. What genes define each cluster?

For each cluster, find the genes that are most specifically high in that cluster versus all others.

In [ ]:
sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon")

for cl in adata.obs["leiden"].cat.categories:
    top_genes = adata.uns["rank_genes_groups"]["names"][cl][:5]
    print(f"cluster {cl:>2} (n={ (adata.obs['leiden']==cl).sum() :>4}):  {', '.join(top_genes)}")

**Walk through 2–3 of these live** — you don't need to read every row out loud:
- One cluster's top genes will be `CD3E`, `CD2`, `TCF7`... → unmistakably **T cells**.
- Another will show `MS4A1`, `CD79A`, `CD19`... → **B cells**.
- Point out one "wow" rare population if you have time — e.g. a cluster marked by `CR2`, `CXCL13`, `VCAM1` is a **follicular dendritic cell** (a stromal cell, not even a classic immune cell, that organizes B cell follicles), or one marked by `CLEC4C`, `IL3RA`, `GZMB` is a **plasmacytoid dendritic cell** — a rare interferon-producing population most people never expect to see pop out of an unsupervised clustering run in a few seconds.

## 6. Naming the clusters

Rather than hand-typing "cluster 4 = B cells" (the exact cluster *numbers* aren't guaranteed to come out the same on every machine — different package versions can shuffle them), we score every cell against a short marker list per candidate cell type, average each score within each cluster, and give each cluster the label it scores highest on. Same "winner-takes-all" logic behind the naming, just automated instead of hardcoded.

In [ ]:
marker_sets = {
    "T cells": ["CD3E", "CD2"],
    "B cells": ["MS4A1", "CD79A", "CD19"],
    "Dendritic cells": ["FSCN1", "LAMP3"],
    "pDCs": ["CLEC4C", "IL3RA", "GZMB"],
    "Macrophages": ["SLC40A1", "MMP9"],
    "Follicular dendritic cells": ["CR2", "CXCL13"],
    "Fibroblastic reticular cells": ["CCL19", "CXCL12"],
    "Endothelial cells": ["PECAM1", "PLVAP"],
}
for name, genes in marker_sets.items():
    sc.tl.score_genes(adata, genes, score_name=f"score_{name}")

score_cols = [f"score_{name}" for name in marker_sets]
cluster_scores = adata.obs.groupby("leiden", observed=True)[score_cols].mean()
cluster_scores.columns = list(marker_sets.keys())
best_label_per_cluster = cluster_scores.idxmax(axis=1)
print(best_label_per_cluster)

adata.obs["cell_type"] = adata.obs["leiden"].map(best_label_per_cluster).astype("category")
adata.obs["cell_type"].value_counts()

In [ ]:
cell_type_palette = {
    "T cells": "#2ECC71",
    "B cells": "#4A90E2",
    "Dendritic cells": "#F1C40F",
    "pDCs": "#E67E22",
    "Macrophages": "#E84393",
    "Follicular dendritic cells": "#9B59B6",
    "Fibroblastic reticular cells": "#95A5A6",
    "Endothelial cells": "#FF3B30",
}

plot_by_boundary("cell_type", cell_type_palette, "cell_type — real cell shapes")

**This is the reveal moment.** Point at the plot: on one side, a dense field of B cells (a *lymphoid follicle* / germinal center) with a handful of follicular dendritic cells and macrophages embedded in it; on the other side, T cells threaded through by fibroblastic reticular cells and blood vessels (a *paracortex* / T cell zone). This is the classic microanatomy of a lymph node — and no one told the algorithm any of it.

## 7. Spatial niches: grouping by microenvironment, not by cell identity

So far we grouped cells by what they *are* (expression → cell type). Now let's group them by *where they sit*: for every cell, look at its 20 nearest spatial neighbors and ask "what mix of cell types surrounds this cell?" Cells with a similar surrounding mix — even if they are different cell types themselves — get grouped into the same **niche**.

In [ ]:
from sklearn.cluster import KMeans

N_NEIGHBORS = 20
N_NICHES = 6

sq.gr.spatial_neighbors(adata, coord_type="generic", n_neighs=N_NEIGHBORS)
adjacency = adata.obsp["spatial_connectivities"].copy()
adjacency.setdiag(1)  # a cell's own type also counts as part of its neighborhood

cell_types = adata.obs["cell_type"].cat.categories.tolist()
onehot = pd.get_dummies(adata.obs["cell_type"]).reindex(columns=cell_types, fill_value=0).values.astype(float)
neighbor_counts = adjacency @ onehot
composition = neighbor_counts / neighbor_counts.sum(axis=1, keepdims=True)

kmeans = KMeans(n_clusters=N_NICHES, random_state=0, n_init=10)
adata.obs["niche"] = pd.Categorical([f"Niche {i}" for i in kmeans.fit_predict(composition)])

adata.obs["niche"].value_counts()

## 8. What is each niche actually made of?

Same idea as the marker-gene table earlier, but for composition instead of genes: average the neighborhood mix within each niche.

In [ ]:
comp_df = pd.DataFrame(composition, columns=cell_types, index=adata.obs_names)
comp_df["niche"] = adata.obs["niche"].values
niche_profile = comp_df.groupby("niche", observed=True)[cell_types].mean()

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(niche_profile.values, aspect="auto", cmap="viridis")
ax.set_xticks(range(len(cell_types)))
ax.set_xticklabels(cell_types, rotation=45, ha="right")
ax.set_yticks(range(len(niche_profile)))
ax.set_yticklabels(niche_profile.index)
plt.colorbar(im, ax=ax, label="mean fraction of neighborhood")
ax.set_title(f"Niche composition ({N_NICHES} niches, {N_NEIGHBORS} spatial neighbors)")
plt.tight_layout()

**Keep this heatmap visible** — you'll need it in a moment to tell which niche number is which, since (just like the leiden cluster numbers earlier) *which* niche gets called "Niche 0" vs "Niche 3" can vary run to run. Read each row: whichever cell type has the brightest cell in that row is what dominates that niche.

Now let's see where these niches actually sit in the tissue.

In [ ]:
niche_cats = sorted(adata.obs["niche"].cat.categories, key=lambda x: int(x.split()[1]))
niche_palette = {cat: plt.cm.tab10(i % 10) for i, cat in enumerate(niche_cats)}

plot_by_boundary("niche", niche_palette, "Spatial niches — real cell shapes")

**Point out the shapes, not just the colors**: a niche dominated by B cells should form a solid core; a mixed B+T niche often forms a *ring* around it (the mantle zone wrapping the germinal center); a niche enriched for endothelial cells tends to trace thin diagonal streaks (a vessel or trabecula running through the tissue); a stromal/fibroblastic-reticular-cell niche often hugs the outer edge of the section (the capsule). None of this was told to the algorithm — it fell out of nothing but "what surrounds each cell."

## 9. Same cell type, two extreme niches — does its state actually change?

This is the real payoff of computing niches instead of just cell types: we can now ask whether a single cell type *behaves* differently depending on where it sits. Let's isolate just the T cells, and pick the two most extreme niches directly from the composition table above: whichever niche has the most B-cell neighbors (T cells buried inside the follicle), and whichever has the most T-cell neighbors (T cells in the purest T-cell territory). Picking niches this way — by their composition, not by typing a niche number — means this still works even if your run labels them differently.

In [ ]:
t_cells = adata[adata.obs["cell_type"] == "T cells"].copy()

niche_B = niche_profile["B cells"].idxmax()
niche_T = niche_profile["T cells"].idxmax()
print(f"Comparing T cells in '{niche_B}' (deepest inside the B follicle) "
      f"vs '{niche_T}' (purest T-cell territory)")

t_sub = t_cells[t_cells.obs["niche"].isin([niche_B, niche_T])].copy()
t_sub.obs["niche"] = t_sub.obs["niche"].cat.remove_unused_categories()
t_sub.obs["niche"].value_counts()

## 10. Where do these two groups of T cells actually sit?

In [ ]:
adata.obs["t_niche_extreme"] = np.nan
adata.obs.loc[t_sub.obs_names[t_sub.obs["niche"] == niche_B], "t_niche_extreme"] = "T cells (B-follicle niche)"
adata.obs.loc[t_sub.obs_names[t_sub.obs["niche"] == niche_T], "t_niche_extreme"] = "T cells (T-core niche)"

highlight_palette = {
    "T cells (B-follicle niche)": "#F1C40F",
    "T cells (T-core niche)": "#4A90E2",
}
plot_by_boundary("t_niche_extreme", highlight_palette, "T cells highlighted by niche — everyone else greyed out")

**Point this out**: the yellow T cells should appear as a sparse scatter *inside* the blue B-cell territory from Section 6, while the blue T cells here form the solid bulk of the paracortex. Same cell type, two completely different addresses.

## 11. What's actually different between them?

A direct two-group comparison — not one-vs-rest — is the classic differential expression design: exactly one question, "B-follicle T cells vs T-core T cells", with a real log fold-change and p-value per gene.

In [ ]:
sc.tl.rank_genes_groups(t_sub, "niche", groups=[niche_B], reference=niche_T, method="wilcoxon")
deg = sc.get.rank_genes_groups_df(t_sub, group=niche_B)

sig = deg[deg["pvals_adj"] < 0.05].copy()
sig = sig.reindex(sig["logfoldchanges"].abs().sort_values(ascending=False).index)
print(f"{len(sig)} / {len(deg)} genes significant (padj < 0.05)")
sig.head(15)[["names", "logfoldchanges", "pvals_adj"]]

In [ ]:
top = sig.head(14).sort_values("logfoldchanges")
colors_bar = ["#F1C40F" if v > 0 else "#4A90E2" for v in top["logfoldchanges"]]

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(top["names"], top["logfoldchanges"], color=colors_bar)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("log2 fold-change (B-follicle niche vs T-core niche)")
ax.set_title("T cells: top genes by niche (padj < 0.05)")
plt.tight_layout()

## 🔎 The insight so far

Two real, well-known immunology signatures fall out of this:

- **T cells in the B-follicle niche light up `CXCL13`, `TOX2`, `PDCD1` (PD-1) and `TIGIT`.** `TOX2` is *the* master transcription factor for **T follicular helper (Tfh)** differentiation, and `CXCL13`/`PDCD1` are textbook Tfh genes.
- **T cells in the T-core niche light up `SELL` (CD62L), `LEF1` and `SATB1`** — classic **naive/quiescent** T cell markers.
- But some of the top "B-follicle" hits (`MS4A1`, `PAX5`, `CD79A`, `CR2`, `CD19`) are themselves B cell genes. Is this contamination, or is it genuine biology — T cells in a follicle *are* there to help B cells, so why not expect some B-activation signature nearby?

## 11b. Telling contamination apart from genuine crosstalk

Don't just assert an answer for a couple of genes — check **every** significant gene, systematically. The key idea: a T cell can only "leak" a transcript it's standing next to. A gene driven purely by physical contamination has to already be expressed by that neighboring cell type — and should fade out with distance from it. A gene driven by a real T cell process doesn't need the neighbor to express it at all.

For genes elevated in the B-follicle niche, the relevant neighbor to check is real B cells (85% of that niche). For genes elevated in the T-core niche, the closest thing to a "contaminating" neighbor is fibroblastic reticular cells — the next most common cell type there after T cells themselves.

In [ ]:
raw = adata.layers["counts"]
gene_idx = {g: i for i, g in enumerate(adata.var_names)}

def pct_positive(gene, cell_names):
    col = raw[adata.obs_names.get_indexer(cell_names), gene_idx[gene]]
    return (np.asarray(col.todense()).flatten() > 0).mean() * 100

b_cell_names = adata.obs_names[adata.obs["cell_type"] == "B cells"]
frc_names = adata.obs_names[adata.obs["cell_type"] == "Fibroblastic reticular cells"]

SPILLOVER_THRESHOLD = 15  # % of the neighboring cell type that must express the gene to call it suspect

neighbor_pct = []
for _, row in sig.iterrows():
    if row["logfoldchanges"] > 0:
        neighbor_pct.append(pct_positive(row["names"], b_cell_names))
    else:
        neighbor_pct.append(pct_positive(row["names"], frc_names))

sig["pct_pos_in_neighbor"] = neighbor_pct
sig["likely_spillover"] = sig["pct_pos_in_neighbor"] > SPILLOVER_THRESHOLD

clean = sig[~sig["likely_spillover"]].copy()
print(f"{len(clean)} / {len(sig)} genes survive the spillover check")
clean[["names", "logfoldchanges", "pvals_adj", "pct_pos_in_neighbor"]].sort_values("logfoldchanges", ascending=False)

**24 of the original 49 genes survive** — the rest (`MS4A1`, `PAX5`, `CD79A`, `CD19`, `CR2`, `CCL19`... ) are exactly the ones with high expression in the relevant physical neighbor, the fingerprint of segmentation spillover, not a T cell discovery. What's left is smaller but trustworthy: on the B-follicle side, `TOX2`, `PDCD1`, `TIGIT`, `MAF` and `SH2D1A` — all genuine, well-documented Tfh genes (`SH2D1A`/SAP in particular is *required* for the physical T cell–B cell interaction that defines Tfh help; mutating it causes a primary immunodeficiency built entirely around this failure). On the T-core side, `TCF7`, `LEF1`, `SATB1`, `SELL`, `CD7` — still a clean naive/quiescent signature.

In [ ]:
top = clean.reindex(clean["logfoldchanges"].abs().sort_values(ascending=False).index).head(16).sort_values("logfoldchanges")
colors_bar = ["#F1C40F" if v > 0 else "#4A90E2" for v in top["logfoldchanges"]]

fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(top["names"], top["logfoldchanges"], color=colors_bar)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("log2 fold-change (B-follicle niche vs T-core niche)")
ax.set_title("T cells: spillover-filtered genes by niche (padj < 0.05)")
plt.tight_layout()

## 12. Zooming out: what pathways do these *clean* genes belong to?

Same idea as before — test whether the gene list as a whole is enriched for any known biological process — but now run only on the 24 genes that survived the spillover check, and, as always, with the panel itself as the background rather than the whole genome.

In [ ]:
import gseapy

up_B_clean = clean[clean["logfoldchanges"] > 0]["names"].tolist()
up_T_clean = clean[clean["logfoldchanges"] < 0]["names"].tolist()
panel_background = adata.var_names.tolist()

enrich_B = gseapy.enrichr(gene_list=up_B_clean, gene_sets=["GO_Biological_Process_2023"],
                           background=panel_background, outdir=None, cutoff=1.0).results
enrich_T = gseapy.enrichr(gene_list=up_T_clean, gene_sets=["GO_Biological_Process_2023"],
                           background=panel_background, outdir=None, cutoff=1.0).results

print(f"UP in B-follicle niche T cells, clean ({len(up_B_clean)} genes):")
sig_B = enrich_B[enrich_B["Adjusted P-value"] < 0.05].sort_values("Adjusted P-value")
print(sig_B[["Term", "Adjusted P-value", "Genes"]].to_string(index=False) if len(sig_B) else "  (no GO term survives correction)")

print(f"\nUP in T-core niche T cells, clean ({len(up_T_clean)} genes):")
sig_T = enrich_T[enrich_T["Adjusted P-value"] < 0.05].sort_values("Adjusted P-value")
print(sig_T[["Term", "Adjusted P-value", "Genes"]].to_string(index=False) if len(sig_T) else "  (no GO term survives correction)")

## 🔎 The final insight — read this one carefully, it's a lesson as much as a result

- **The flashy "B cell activation" pathway hit from before is gone.** That's not a failure — it's confirmation that it was never real to begin with: it was built entirely out of the genes Section 11b just removed. With those gone, 11 genuinely T-cell-expressed genes (`TOX2`, `PDCD1`, `TIGIT`, `MAF`, `SH2D1A`...) aren't numerous enough, or don't share a common enough GO annotation, to clear multiple-testing correction as a pathway. That doesn't mean they're not real — we already validated each of them individually in Section 11b. It means pathway enrichment needs a certain density of genes sharing an annotation, which a short, clean, targeted-panel gene list often won't have.
- The T-core side still shows a modest, coherent signal — interleukin-4 response (`LEF1`, `TCF7`) — consistent with naive T cell homeostasis, on a much smaller and more trustworthy gene list than before.

**Takeaway for your own data, beyond this demo**: don't chase a pretty pathway plot before checking what's driving it. A big, significant-looking enrichment built on contaminated genes is worse than no enrichment at all — it looks like a discovery and isn't one. A short, honestly-filtered gene list that doesn't reach pathway-level significance is a more trustworthy result than a long, dirty one that does.

## (Bonus — only if time allows)

- Re-run the clustering at `resolution=0.5` or `resolution=1.5` and see which of today's clusters merge or split.
- Repeat Sections 9–12 on `cell_type == "B cells"` instead of T cells — do B cells also show a niche-dependent state?
- Look at one of the `score_*` columns computed earlier directly, per cell instead of averaged per cluster: `sq.pl.spatial_scatter(adata, color="score_B cells", shape=None, size=6)` — a continuous alternative to discrete cluster labels.